# Walmart Business Questions â€” My SQL Solutions

Each section has your saved SQL solution, followed by its output and a space for your own conclusion.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from IPython.display import display, HTML

load_dotenv()
engine = create_engine(os.environ['DATABASE_URL'])

def run_sql(sql):
    with engine.connect() as connection:
        result = connection.execute(text(sql))
        data = pd.DataFrame(result.fetchall(), columns=result.keys())
    display(HTML('''<style>.sql-results table{border-collapse:collapse;color:#f9fafb}.sql-results th{background:#1f2937;padding:8px}.sql-results td{background:#111827;padding:8px}.sql-results th,.sql-results td{border:1px solid #374151}</style><div class="sql-results">''' + data.to_html(index=False) + '</div>'))

print('Connected to PostgreSQL. Run any question below.')

Connected to PostgreSQL. Run any question below.


## Business Question 1
Find payment methods, transaction count, and quantities sold.

In [2]:
sql = '''
select payment_method, count(invoice_id) as no_of_transaction , sum(quantity) as total_quantity from walmart
group by payment_method;
'''
run_sql(sql)

payment_method,no_of_transaction,total_quantity
Credit card,4256,9567.0
Ewallet,3881,8932.0
Cash,1832,4984.0


**My notes / conclusion:** 


### Analysis

- Credit Card has the highest number of transactions (4,256), followed by Ewallet (3,881) and Cash (1,832).
- Credit Card also has the highest total quantity sold (9,567), followed by Ewallet (8,932) and Cash (4,984).
- Cash has the lowest transaction volume but the highest average quantity per transaction.

### Business Insight

The results indicate a strong preference for cashless payment methods, with Credit Card and Ewallet accounting for most transactions. Although Cash has fewer transactions, customers using Cash purchase more items per transaction on average (approximately 2.72 items).

### Limitation

The dataset shows which payment methods are used but does not explain why customers prefer a particular payment method. Therefore, reasons such as convenience, rewards, or credit availability should be treated as possible explanations rather than confirmed findings.

## Business Question 2
Find the highest-rated category in each branch.

In [ ]:
sql = '''
SELECT *
FROM (
    SELECT
        branch,
        category,
        ROUND(CAST(AVG(rating) AS numeric), 2) AS avg_rating,
        ROW_NUMBER() OVER (
            PARTITION BY branch
            ORDER BY AVG(rating) DESC
        ) AS rown
    FROM walmart
    GROUP BY branch, category
) t
WHERE rown = 1
LIMIT 10;
'''
run_sql(sql)

branch,category,avg_rating,rown
WALM001,Electronic accessories,7.45,1
WALM002,Food and beverages,8.25,1
WALM003,Sports and travel,7.50,1
WALM004,Food and beverages,9.30,1
WALM005,Health and beauty,8.37,1
WALM006,Fashion accessories,6.80,1
WALM007,Food and beverages,7.55,1
WALM008,Food and beverages,7.40,1
WALM009,Sports and travel,9.60,1
WALM010,Electronic accessories,9.00,1


**My notes / conclusion:** 


### Analysis

- The highest-rated category varies across branches, with Health and beauty, Sports and travel, Food and beverages, and Electronic accessories appearing frequently as the top-rated categories.
- WALM034 has the highest branch-level average rating at 10.00 for Health and beauty.
- The lowest average rating among the selected top-rated categories is 5.28 for Electronic accessories at WALM050.
- This shows that customer ratings differ considerably across branches and their leading categories.

### Business Insight

Customer satisfaction varies by branch and category. Health and beauty, Sports and travel, Food and beverages, and Electronic accessories are frequently the highest-rated categories across branches. This suggests that customer satisfaction is not uniform across the store network and may vary based on the branch-category combination.

### Limitation

This analysis only identifies the highest-rated category within each branch. It does not establish the overall best category across Walmart or explain why ratings differ between branches. Additional product-level and customer-feedback data would be required to investigate the reasons behind these differences.

## Business Question 3
Find the busiest day for each branch.

In [ ]:
sql = '''
select * from(
SELECT 
branch ,
to_char(date,'day') as day_name,
count(invoice_id) as total_transaction,
row_number() over(partition by branch order by count(invoice_id)desc) as rown
from walmart
group by branch,to_char(date,'day')
)t
where rown =1;
'''
run_sql(sql)

**My notes / conclusion:** 


## Business Question 4
Find total quantity sold per payment method.

In [ ]:
sql = '''
select payment_method,sum(quantity) as quantity_sold from walmart
group by payment_method
'''
run_sql(sql)

**My notes / conclusion:** 


## Business Question 5
Find average, minimum, and maximum rating per city.

In [ ]:
sql = '''
SELECT city, 
       ROUND(CAST(AVG(rating) AS numeric), 2) AS avg_rating, 
       MIN(rating) AS min_rating, 
       MAX(rating) AS max_rating 
FROM walmart
GROUP BY city;
'''
run_sql(sql)

**My notes / conclusion:** 


## Business Question 6
Find total profit per category.

In [ ]:
sql = '''
SELECT 
    category, 
    ROUND(cast(SUM(unit_price * quantity * profit_margin)as numeric),3) AS total_profit
FROM walmart
GROUP BY category
ORDER BY total_profit DESC;
'''
run_sql(sql)

**My notes / conclusion:** 


## Business Question 7
Find preferred payment method per branch.

In [ ]:
sql = '''
select branch,payment_method from (
SELECT *,
       ROW_NUMBER() OVER (
           PARTITION BY branch
           ORDER BY transac_pb DESC
       ) AS rn
FROM (
    SELECT 
        branch,
        payment_method,
        COUNT(payment_method) AS transac_pb
    FROM walmart
    GROUP BY branch, payment_method
) t
) x
where rn=1;
'''
run_sql(sql)

**My notes / conclusion:** 


## Business Question 8
Categorize transactions into morning, afternoon, and evening shifts.

In [ ]:
sql = '''
 select
 case
   when time<'12:00:00' then 'morning'
   when time<'17:00:00' then 'afternoon'
   else  'Evening'
  End as shift,count (invoice_id) as no_trans
  from walmart
  group by shift
 order by no_trans desc
'''
run_sql(sql)

**My notes / conclusion:** 
